In [0]:
# ==========================================================
# Load Gold Layer: Databricks -> Snowflake
#
# Updated Gold tables:
#   DIM_BENEFICIARY
#   FACT_CLAIMS
#   FACT_INPATIENT
#   FACT_PHARMACY
#
# Existing unchanged dimensions are also loaded so that
# Snowflake remains a complete copy of Databricks Gold.
# ==========================================================

# ==========================================================
# SNOWFLAKE CONNECTION
# ==========================================================

sf_options = {
    "host": "MDHABOF-MT28339.snowflakecomputing.com",

    "sfUser": "VAIBHAVX14",

    "sfPassword": dbutils.secrets.get(
        scope="claims-snowflake",
        key="snowflake-password"
    ),

    "sfDatabase": "HEALTHCARE_CLAIMS_DB",
    "sfSchema": "GOLD",
    "sfWarehouse": "CLAIMS_WH",
    "sfRole": "ACCOUNTADMIN",
}


# ==========================================================
# GOLD TABLE MAPPING
# ==========================================================

GOLD_TABLES = {

    # Dimensions
    "dim_beneficiary": "DIM_BENEFICIARY",
    "dim_provider": "DIM_PROVIDER",
    "dim_date": "DIM_DATE",
    "dim_diagnosis": "DIM_DIAGNOSIS",
    "dim_claim_status": "DIM_CLAIM_STATUS",

    # Facts
    "fact_claims": "FACT_CLAIMS",
    "fact_inpatient": "FACT_INPATIENT",
    "fact_pharmacy": "FACT_PHARMACY",

}


CATALOG = "healthcare_claims_catalog"
SCHEMA = "gold"


# ==========================================================
# RESULTS
# ==========================================================

results = []


# ==========================================================
# LOAD GOLD TABLES
# ==========================================================

for db_table, sf_table in GOLD_TABLES.items():

    full_db_table = f"{CATALOG}.{SCHEMA}.{db_table}"

    print("\n" + "-" * 70)
    print(f"Processing: {full_db_table}")
    print(f"Target    : {sf_table}")
    print("-" * 70)

    try:

        # --------------------------------------------------
        # Read Databricks Gold
        # --------------------------------------------------

        df = spark.table(full_db_table)

        databricks_count = df.count()

        print(
            f"Databricks rows: {databricks_count:,}"
        )


        # --------------------------------------------------
        # Write to Snowflake
        # --------------------------------------------------
        #
        # Gold tables are full snapshot rebuilds.
        # Therefore overwrite is intentional.
        #

        (
            df.write
            .format("snowflake")
            .options(**sf_options)
            .option("dbtable", sf_table)
            .mode("overwrite")
            .save()
        )

        print(
            f"Loaded into Snowflake: {sf_table}"
        )


        # --------------------------------------------------
        # Read Snowflake Count
        # --------------------------------------------------

        sf_count_df = (

            spark.read

            .format("snowflake")

            .options(**sf_options)

            .option(
                "query",
                f"""
                SELECT COUNT(*) AS CNT
                FROM {sf_table}
                """
            )

            .load()

        )


        snowflake_count = (
            sf_count_df
            .collect()[0]["CNT"]
        )


        # --------------------------------------------------
        # Reconciliation
        # --------------------------------------------------

        if databricks_count == snowflake_count:

            status = "PASS"

            print(
                f"[PASS] "
                f"Databricks={databricks_count:,} | "
                f"Snowflake={snowflake_count:,}"
            )

        else:

            status = "MISMATCH"

            print(
                f"[MISMATCH] "
                f"Databricks={databricks_count:,} | "
                f"Snowflake={snowflake_count:,}"
            )


        results.append(
            (
                db_table,
                sf_table,
                databricks_count,
                snowflake_count,
                status
            )
        )


    except Exception as e:

        status = f"ERROR: {str(e)}"

        print(
            f"[ERROR] {db_table} -> {sf_table}"
        )

        print(
            str(e)
        )

        results.append(
            (
                db_table,
                sf_table,
                None,
                None,
                status
            )
        )


# ==========================================================
# SUMMARY
# ==========================================================

print("\n")
print("=" * 80)
print("GOLD LAYER LOAD SUMMARY")
print("Databricks -> Snowflake")
print("=" * 80)


summary_df = spark.createDataFrame(
    results,
    schema="""
        databricks_table string,
        snowflake_table string,
        databricks_count long,
        snowflake_count long,
        status string
    """
)


display(summary_df)


# ==========================================================
# FINAL VALIDATION
# ==========================================================

failures = [

    r for r in results
    if r[4] != "PASS"

]


print("\n" + "=" * 80)


if failures:

    print(
        f"FAILED: {len(failures)} table(s) "
        "did not pass reconciliation."
    )

    print(
        "Do NOT proceed to BI validation until the mismatches are resolved."
    )

else:

    print(
        "SUCCESS: All 8 Gold tables loaded successfully."
    )

    print(
        "Databricks and Snowflake row counts match."
    )

    print(
        "Snowflake Gold is ready for BI validation."
    )


print("=" * 80)


----------------------------------------------------------------------
Processing: healthcare_claims_catalog.gold.dim_beneficiary
Target    : DIM_BENEFICIARY
----------------------------------------------------------------------
Databricks rows: 116,370
Loaded into Snowflake: DIM_BENEFICIARY
[PASS] Databricks=116,370 | Snowflake=116,370

----------------------------------------------------------------------
Processing: healthcare_claims_catalog.gold.dim_provider
Target    : DIM_PROVIDER
----------------------------------------------------------------------
Databricks rows: 620,594
Loaded into Snowflake: DIM_PROVIDER
[PASS] Databricks=620,594 | Snowflake=620,594

----------------------------------------------------------------------
Processing: healthcare_claims_catalog.gold.dim_date
Target    : DIM_DATE
----------------------------------------------------------------------
Databricks rows: 2,021
Loaded into Snowflake: DIM_DATE
[PASS] Databricks=2,021 | Snowflake=2,021

---------------

databricks_table,snowflake_table,databricks_count,snowflake_count,status
dim_beneficiary,DIM_BENEFICIARY,116370,116370,PASS
dim_provider,DIM_PROVIDER,620594,620594,PASS
dim_date,DIM_DATE,2021,2021,PASS
dim_diagnosis,DIM_DIAGNOSIS,13420,13420,PASS
dim_claim_status,DIM_CLAIM_STATUS,21,21,PASS
fact_claims,FACT_CLAIMS,11073217,11073217,PASS
fact_inpatient,FACT_INPATIENT,66165,66165,PASS
fact_pharmacy,FACT_PHARMACY,16540128,16540128,PASS



SUCCESS: All 8 Gold tables loaded successfully.
Databricks and Snowflake row counts match.
Snowflake Gold is ready for BI validation.
